<a href="https://colab.research.google.com/github/yujiunzou/supply-chain-analytics/blob/main/01_data_profiling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from pathlib import Path

print("pandas version:", pd.__version__)

pandas version: 2.2.3


In [2]:
data_folder = Path("/content")

csv_files = list(data_folder.glob("*.csv"))

print("CSV files found:", len(csv_files))

for file in csv_files:
    print(file.name)

CSV files found: 7
vmi_customers.csv
freight_rates.csv
products_per_plant.csv
plant_ports.csv
warehouse_costs.csv
warehouse_capacities.csv
orders.csv


# 1.Data Profiling

In [25]:
def profile_data(df, table_name):
    print("=" * 70)
    print(f"TABLE: {table_name}")
    print("=" * 70)

    # 1. Dataset size
    print("\n1. Shape")
    print(f"Rows: {df.shape[0]}")
    print(f"Columns: {df.shape[1]}")

    # 2. First five rows
    print("\n2. First 5 rows")
    display(df.head())

    # 3. Column information
    print("\n3. Data types and non-null counts")
    df.info()

    # 4. Descriptive statistics
    print("\n4. Descriptive statistics")
    display(df.describe(include="all").T)

    # 5. Standard missing values
    print("\n5. Standard missing values")
    display(
        df.isna()
          .sum()
          .to_frame("missing_count")
    )

    # 6. Hidden missing values
    missing_labels = {
        "",
        "null",
        "none",
        "n/a",
        "na",
        "unknown",
        "-"
    }

    hidden_missing = {}

    for column in df.select_dtypes(include="object").columns:
        normalized = (
            df[column]
            .astype(str)
            .str.strip()
            .str.lower()
        )

        hidden_missing[column] = (
            normalized.isin(missing_labels).sum()
        )

    print("\n6. Possible hidden missing values")

    if hidden_missing:
        display(
            pd.Series(
                hidden_missing,
                name="hidden_missing_count"
            ).to_frame()
        )
    else:
        print("No text columns to check.")

    # 7. Duplicate rows
    duplicate_count = df.duplicated().sum()

    print("\n7. Duplicate rows")
    print(f"Duplicate count: {duplicate_count}")

    if duplicate_count > 0:
        display(df[df.duplicated(keep=False)])

    # 8. Unique values per column
    print("\n8. Unique values per column")
    display(
        df.nunique(dropna=False)
          .to_frame("unique_count")
    )

## 1.1 Orders


*   order_id: Order ID is ID of the order made by the customer.
*   product_id: Product ID is the specific product ID customer ordered.
*   tpt_day_cnt: Estimated shipping time.



In [26]:
orders = pd.read_csv("/content/orders.csv")
profile_data(orders, "Orders")

TABLE: Orders

1. Shape
Rows: 9215
Columns: 14

2. First 5 rows


,order_id,order_date,origin_port,carrier,tpt,service_level,ship_ahead_day_count,ship_late_day_count,customer,product_id,plant_code,destination_port,unit_quantity,weight
0,1.447296e+09,2013-05-26,PORT09,V44_3,1,CRF,3,0,V55555_53,1700106,PLANT16,PORT09,808,14.30
1,1.447158e+09,2013-05-26,PORT09,V44_3,1,CRF,3,0,V55555_53,1700106,PLANT16,PORT09,3188,87.94
2,1.447139e+09,2013-05-26,PORT09,V44_3,1,CRF,3,0,V55555_53,1700106,PLANT16,PORT09,2331,61.20
3,1.447364e+09,2013-05-26,PORT09,V44_3,1,CRF,3,0,V55555_53,1700106,PLANT16,PORT09,847,16.16
4,1.447364e+09,2013-05-26,PORT09,V44_3,1,CRF,3,0,V55555_53,1700106,PLANT16,PORT09,2163,52.34



3. Data types and non-null counts
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9215 entries, 0 to 9214
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   order_id              9215 non-null   float64
 1   order_date            9215 non-null   object 
 2   origin_port           9215 non-null   object 
 3   carrier               9215 non-null   object 
 4   tpt                   9215 non-null   int64  
 5   service_level         9215 non-null   object 
 6   ship_ahead_day_count  9215 non-null   int64  
 7   ship_late_day_count   9215 non-null   int64  
 8   customer              9215 non-null   object 
 9   product_id            9215 non-null   int64  
 10  plant_code            9215 non-null   object 
 11  destination_port      9215 non-null   object 
 12  unit_quantity         9215 non-null   int64  
 13  weight                9215 non-null   float64
dtypes: float64(2), int64(5), object(7)
me

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
order_id,9215.0,NaN,NaN,NaN,1447273698.881769,83816.29279,1447125805.7,1447196681.2,1447275684.7,1447346498.7,1447425230.7
order_date,9215,1,2013-05-26,9215,NaN,NaN,NaN,NaN,NaN,NaN,NaN
origin_port,9215,3,PORT04,9041,NaN,NaN,NaN,NaN,NaN,NaN,NaN
carrier,9215,3,V444_0,6264,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tpt,9215.0,NaN,NaN,NaN,1.717743,0.6305,0.0,1.0,2.0,2.0,4.0
service_level,9215,3,DTP,6218,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ship_ahead_day_count,9215.0,NaN,NaN,NaN,1.852306,1.922302,0.0,0.0,3.0,3.0,6.0
ship_late_day_count,9215.0,NaN,NaN,NaN,0.039935,0.319625,0.0,0.0,0.0,0.0,6.0
customer,9215,46,V55555555555555_8,976,NaN,NaN,NaN,NaN,NaN,NaN,NaN
product_id,9215.0,NaN,NaN,NaN,1680536.058709,15265.927778,1613321.0,1669702.0,1683636.0,1689554.0,1702654.0



5. Standard missing values


,missing_count
order_id,0
order_date,0
origin_port,0
carrier,0
tpt,0
service_level,0
ship_ahead_day_count,0
ship_late_day_count,0
customer,0
product_id,0



6. Possible hidden missing values


,hidden_missing_count
order_date,0
origin_port,0
carrier,0
service_level,0
customer,0
plant_code,0
destination_port,0



7. Duplicate rows
Duplicate count: 0

8. Unique values per column


,unique_count
order_id,9215
order_date,1
origin_port,3
carrier,3
tpt,5
service_level,3
ship_ahead_day_count,5
ship_late_day_count,4
customer,46
product_id,772


## 1.2 Freight Rate

*   tpt_day_cnt: Estimated shipping time.

In [27]:
freight_rates = pd.read_csv("/content/freight_rates.csv")
profile_data(freight_rates, "Freight Rates")

TABLE: Freight Rates

1. Shape
Rows: 1540
Columns: 11

2. First 5 rows


,carrier,orig_port_cd,dest_port_cd,minm_wgh_qty,max_wgh_qty,svc_cd,minimum_cost,rate,mode_dsc,tpt_day_cnt,carrier_type
0,V444_6,PORT08,PORT09,250.0,499.99,DTD,43.2272,0.7132,AIR,2,V88888888_0
1,V444_6,PORT08,PORT09,65.0,69.99,DTD,43.2272,0.7512,AIR,2,V88888888_0
2,V444_6,PORT08,PORT09,60.0,64.99,DTD,43.2272,0.7892,AIR,2,V88888888_0
3,V444_6,PORT08,PORT09,50.0,54.99,DTD,43.2272,0.8272,AIR,2,V88888888_0
4,V444_6,PORT08,PORT09,35.0,39.99,DTD,43.2272,1.0552,AIR,2,V88888888_0



3. Data types and non-null counts
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1540 entries, 0 to 1539
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   carrier       1540 non-null   object 
 1   orig_port_cd  1540 non-null   object 
 2   dest_port_cd  1540 non-null   object 
 3   minm_wgh_qty  1540 non-null   float64
 4   max_wgh_qty   1540 non-null   float64
 5   svc_cd        1540 non-null   object 
 6   minimum_cost  1540 non-null   float64
 7   rate          1540 non-null   float64
 8   mode_dsc      1540 non-null   object 
 9   tpt_day_cnt   1540 non-null   int64  
 10  carrier_type  1540 non-null   object 
dtypes: float64(4), int64(1), object(6)
memory usage: 132.5+ KB

4. Descriptive statistics


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
carrier,1540,9,V444_4,906,NaN,NaN,NaN,NaN,NaN,NaN,NaN
orig_port_cd,1540,10,PORT06,479,NaN,NaN,NaN,NaN,NaN,NaN,NaN
dest_port_cd,1540,1,PORT09,1540,NaN,NaN,NaN,NaN,NaN,NaN,NaN
minm_wgh_qty,1540.0,NaN,NaN,NaN,155.886637,475.952049,0.0,15.01,41.01,67.229614,10000.0
max_wgh_qty,1540.0,NaN,NaN,NaN,4636.143633,20251.309727,0.453592,21.5,47.5,99.99,99999.99
svc_cd,1540,2,DTD,1322,NaN,NaN,NaN,NaN,NaN,NaN,NaN
minimum_cost,1540.0,NaN,NaN,NaN,12.329033,22.946192,1.202,3.6568,7.417,11.4945,425.0272
rate,1540.0,NaN,NaN,NaN,2.892656,4.603877,0.0332,0.4704,1.6612,3.9322,128.0272
mode_dsc,1540,2,AIR,1511,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tpt_day_cnt,1540.0,NaN,NaN,NaN,2.186364,2.000895,0.0,1.0,2.0,2.0,14.0



5. Standard missing values


...

,hidden_missing_count
carrier,0
orig_port_cd,0
dest_port_cd,0
svc_cd,0
mode_dsc,0
carrier_type,0



7. Duplicate rows
Duplicate count: 3


,carrier,orig_port_cd,dest_port_cd,minm_wgh_qty,max_wgh_qty,svc_cd,minimum_cost,rate,mode_dsc,tpt_day_cnt,carrier_type
1492,V444_5,PORT03,PORT09,0.0,5000.0,DTD,27.2024,9.2024,GROUND,1,V88888888_0
1494,V444_5,PORT03,PORT09,0.0,5000.0,DTD,27.2024,9.2024,GROUND,1,V88888888_0
1504,V444_0,PORT03,PORT09,0.0,5000.0,DTD,27.2024,13.2024,GROUND,1,V88888888_0
1505,V444_0,PORT03,PORT09,0.0,5000.0,DTD,27.2024,13.2024,GROUND,1,V88888888_0
1510,V444_0,PORT03,PORT09,0.0,5000.0,DTD,27.2024,13.2024,GROUND,1,V88888888_0



8. Unique values per column


,unique_count
carrier,9
orig_port_cd,10
dest_port_cd,1
minm_wgh_qty,327
max_wgh_qty,327
svc_cd,2
minimum_cost,787
rate,1028
mode_dsc,2
tpt_day_cnt,8


## 1.3 Plant Ports

In [28]:
plant_ports = pd.read_csv("/content/plant_ports.csv")
profile_data(plant_ports, "Plant Ports")

TABLE: Plant Ports

1. Shape
Rows: 22
Columns: 2

2. First 5 rows


,plant_code,port
0,PLANT01,PORT01
1,PLANT01,PORT02
2,PLANT02,PORT03
3,PLANT03,PORT04
4,PLANT04,PORT05



3. Data types and non-null counts
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22 entries, 0 to 21
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   plant_code  22 non-null     object
 1   port        22 non-null     object
dtypes: object(2)
memory usage: 484.0+ bytes

4. Descriptive statistics


,count,unique,top,freq
plant_code,22,19,PLANT01,2
port,22,11,PORT04,7



5. Standard missing values


,missing_count
plant_code,0
port,0



6. Possible hidden missing values


,hidden_missing_count
plant_code,0
port,0



7. Duplicate rows
Duplicate count: 0

8. Unique values per column


,unique_count
plant_code,19
port,11


## 1.4 Products per Plant


In [17]:
products_per_plant = pd.read_csv("/content/products_per_plant.csv")
profile_data(products_per_plant, "Products per Plant")

TABLE: Products per Plant
Rows: 2036
Columns: 2

Column names:
['plant_code', 'product_id']

Data types:
plant_code    object
product_id     int64
dtype: object

Standard missing values:
plant_code    0
product_id    0
dtype: int64

Possible hidden missing values:
plant_code    0
dtype: int64

Duplicate rows:
0

First 5 rows:


,plant_code,product_id
0,PLANT15,1698815
1,PLANT17,1664419
2,PLANT17,1664426
3,PLANT17,1672826
4,PLANT17,1674916


## 1.5 VMI Customers

In [18]:
vmi_customers = pd.read_csv("/content/vmi_customers.csv")
profile_data(vmi_customers, "VMI Customers")

TABLE: VMI Customers
Rows: 14
Columns: 2

Column names:
['plant_code', 'customers']

Data types:
plant_code    object
customers     object
dtype: object

Standard missing values:
plant_code    0
customers     0
dtype: int64

Possible hidden missing values:
plant_code    0
customers     0
dtype: int64

Duplicate rows:
0

First 5 rows:


,plant_code,customers
0,PLANT02,V5555555555555_16
1,PLANT02,V555555555555555_29
2,PLANT02,V555555555_3
3,PLANT02,V55555555555555_8
4,PLANT02,V55555555_9


## 1.6 Warehouse Capacities

WhCapacities table is the maximum number of orders that can be processed per each plant, it is not dependant on specific products.

代表每個 plant 每天最多能處理的訂單筆數，不是產品件數，也不是重量。
例如：Customer 1 訂購 10 件 Product X

In [19]:
warehouse_capacities = pd.read_csv(
    "/content/warehouse_capacities.csv"
)
profile_data(warehouse_capacities, "Warehouse Capacities")

TABLE: Warehouse Capacities
Rows: 19
Columns: 2

Column names:
['plant_id', 'daily_capacity']

Data types:
plant_id          object
daily_capacity     int64
dtype: object

Standard missing values:
plant_id          0
daily_capacity    0
dtype: int64

Possible hidden missing values:
plant_id    0
dtype: int64

Duplicate rows:
0

First 5 rows:


,plant_id,daily_capacity
0,PLANT15,11
1,PLANT17,8
2,PLANT18,111
3,PLANT05,385
4,PLANT02,138


## 1.7 Warehouse Costs

In [20]:
warehouse_costs = pd.read_csv("/content/warehouse_costs.csv")
profile_data(warehouse_costs, "Warehouse Costs")

TABLE: Warehouse Costs
Rows: 19
Columns: 2

Column names:
['wh', 'cost_unit']

Data types:
wh            object
cost_unit    float64
dtype: object

Standard missing values:
wh           0
cost_unit    0
dtype: int64

Possible hidden missing values:
wh    0
dtype: int64

Duplicate rows:
0

First 5 rows:


,wh,cost_unit
0,PLANT15,1.415063
1,PLANT17,0.428947
2,PLANT18,2.036254
3,PLANT05,0.488144
4,PLANT02,0.477504


#2.Data quality checks

In [21]:
tables = {
    "orders": orders,
    "freight_rates": freight_rates,
    "plant_ports": plant_ports,
    "products_per_plant": products_per_plant,
    "vmi_customers": vmi_customers,
    "warehouse_capacities": warehouse_capacities,
    "warehouse_costs": warehouse_costs
}

In [22]:
quality_summary = []

for table_name, df in tables.items():
    quality_summary.append({
        "table": table_name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "missing_values": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum())
    })

quality_summary = pd.DataFrame(quality_summary)
quality_summary

,table,rows,columns,missing_values,duplicate_rows
0,orders,9215,14,0,0
1,freight_rates,1540,11,0,3
2,plant_ports,22,2,0,0
3,products_per_plant,2036,2,0,0
4,vmi_customers,14,2,0,0
5,warehouse_capacities,19,2,0,0
6,warehouse_costs,19,2,0,0


## 2.1 Duplicate Records 重複資料

In [23]:
for table_name, df in tables.items():
    duplicate_count = df.duplicated().sum()

    print(f"{table_name}: {duplicate_count} duplicate rows")

    if duplicate_count > 0:
        display(df[df.duplicated(keep=False)].sort_values(
            by=df.columns.tolist()
        ))

orders: 0 duplicate rows
freight_rates: 3 duplicate rows


,carrier,orig_port_cd,dest_port_cd,minm_wgh_qty,max_wgh_qty,svc_cd,minimum_cost,rate,mode_dsc,tpt_day_cnt,carrier_type
1504,V444_0,PORT03,PORT09,0.0,5000.0,DTD,27.2024,13.2024,GROUND,1,V88888888_0
1505,V444_0,PORT03,PORT09,0.0,5000.0,DTD,27.2024,13.2024,GROUND,1,V88888888_0
1510,V444_0,PORT03,PORT09,0.0,5000.0,DTD,27.2024,13.2024,GROUND,1,V88888888_0
1492,V444_5,PORT03,PORT09,0.0,5000.0,DTD,27.2024,9.2024,GROUND,1,V88888888_0
1494,V444_5,PORT03,PORT09,0.0,5000.0,DTD,27.2024,9.2024,GROUND,1,V88888888_0


plant_ports: 0 duplicate rows
products_per_plant: 0 duplicate rows
vmi_customers: 0 duplicate rows
warehouse_capacities: 0 duplicate rows
warehouse_costs: 0 duplicate rows


##2.2 Numeric Value Validation 數值欄位合理性檢查

In [24]:
for table_name, df in tables.items():
    print("\n" + "=" * 60)
    print(f"NUMERIC SUMMARY: {table_name}")
    print("=" * 60)

    numeric_columns = df.select_dtypes(include="number")

    if numeric_columns.empty:
        print("No numeric columns.")
    else:
        display(numeric_columns.describe().T)


NUMERIC SUMMARY: orders


,count,mean,std,min,25%,50%,75%,max
order_id,9215.0,1.447274e+09,83816.292790,1.447126e+09,1.447197e+09,1.447276e+09,1.447346e+09,1.447425e+09
tpt,9215.0,1.717743e+00,0.630500,0.000000e+00,1.000000e+00,2.000000e+00,2.000000e+00,4.000000e+00
ship_ahead_day_count,9215.0,1.852306e+00,1.922302,0.000000e+00,0.000000e+00,3.000000e+00,3.000000e+00,6.000000e+00
ship_late_day_count,9215.0,3.993489e-02,0.319625,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,6.000000e+00
product_id,9215.0,1.680536e+06,15265.927778,1.613321e+06,1.669702e+06,1.683636e+06,1.689554e+06,1.702654e+06
unit_quantity,9215.0,3.202747e+03,15965.622260,2.350000e+02,3.300000e+02,4.770000e+02,1.275500e+03,5.618470e+05
weight,9215.0,1.987169e+01,66.569064,0.000000e+00,1.407430e+00,4.440000e+00,1.332567e+01,2.338405e+03



NUMERIC SUMMARY: freight_rates


,count,mean,std,min,25%,50%,75%,max
minm_wgh_qty,1540.0,155.886637,475.952049,0.000000,15.0100,41.0100,67.229614,10000.0000
max_wgh_qty,1540.0,4636.143633,20251.309727,0.453592,21.5000,47.5000,99.990000,99999.9900
minimum_cost,1540.0,12.329033,22.946192,1.202000,3.6568,7.4170,11.494500,425.0272
rate,1540.0,2.892656,4.603877,0.033200,0.4704,1.6612,3.932200,128.0272
tpt_day_cnt,1540.0,2.186364,2.000895,0.000000,1.0000,2.0000,2.000000,14.0000



NUMERIC SUMMARY: plant_ports
No numeric columns.

NUMERIC SUMMARY: products_per_plant


,count,mean,std,min,25%,50%,75%,max
product_id,2036.0,1.684180e+06,13148.043792,1613321.0,1680697.0,1685269.5,1692998.75,1702654.0



NUMERIC SUMMARY: vmi_customers
No numeric columns.

NUMERIC SUMMARY: warehouse_capacities


,count,mean,std,min,25%,50%,75%,max
daily_capacity,19.0,304.789474,323.51328,7.0,31.5,209.0,473.5,1070.0



NUMERIC SUMMARY: warehouse_costs


,count,mean,std,min,25%,50%,75%,max
cost_unit,19.0,0.7241,0.495091,0.371424,0.473605,0.522857,0.637046,2.036254


## 2.3 Key Uniqueness

In [29]:
def check_key_uniqueness(df, table_name, key_columns):
    total_rows = len(df)

    duplicate_mask = df.duplicated(
        subset=key_columns,
        keep=False
    )

    duplicate_rows = duplicate_mask.sum()

    unique_keys = (
        df[key_columns]
        .drop_duplicates()
        .shape[0]
    )

    is_unique = duplicate_rows == 0

    print("=" * 70)
    print(f"TABLE: {table_name}")
    print(f"CANDIDATE KEY: {key_columns}")
    print("=" * 70)

    print(f"Total rows: {total_rows}")
    print(f"Unique key combinations: {unique_keys}")
    print(f"Rows involved in duplicate keys: {duplicate_rows}")
    print(f"Is key unique? {is_unique}")

    if duplicate_rows > 0:
        print("\nDuplicate key records:")
        display(
            df.loc[duplicate_mask]
              .sort_values(key_columns)
        )

    return {
        "table": table_name,
        "candidate_key": ", ".join(key_columns),
        "total_rows": total_rows,
        "unique_keys": unique_keys,
        "duplicate_key_rows": int(duplicate_rows),
        "is_unique": is_unique
    }

In [30]:
key_results = []

key_results.append(
    check_key_uniqueness(
        orders,
        "Orders",
        ["order_id"]
    )
)

key_results.append(
    check_key_uniqueness(
        freight_rates,
        "Freight Rates",
        [
            "carrier",
            "orig_port_cd",
            "dest_port_cd",
            "minm_wgh_qty",
            "max_wgh_qty",
            "svc_cd"
        ]
    )
)

key_results.append(
    check_key_uniqueness(
        plant_ports,
        "Plant Ports",
        ["plant_code", "port"]
    )
)

key_results.append(
    check_key_uniqueness(
        products_per_plant,
        "Products per Plant",
        ["plant_code", "product_id"]
    )
)

key_results.append(
    check_key_uniqueness(
        vmi_customers,
        "VMI Customers",
        ["plant_code", "customers"]
    )
)

key_results.append(
    check_key_uniqueness(
        warehouse_capacities,
        "Warehouse Capacities",
        ["plant_id"]
    )
)

key_results.append(
    check_key_uniqueness(
        warehouse_costs,
        "Warehouse Costs",
        ["wh"]
    )
)

TABLE: Orders
CANDIDATE KEY: ['order_id']
Total rows: 9215
Unique key combinations: 9215
Rows involved in duplicate keys: 0
Is key unique? True
TABLE: Freight Rates
CANDIDATE KEY: ['carrier', 'orig_port_cd', 'dest_port_cd', 'minm_wgh_qty', 'max_wgh_qty', 'svc_cd']
Total rows: 1540
Unique key combinations: 1258
Rows involved in duplicate keys: 414
Is key unique? False

Duplicate key records:


,carrier,orig_port_cd,dest_port_cd,minm_wgh_qty,max_wgh_qty,svc_cd,minimum_cost,rate,mode_dsc,tpt_day_cnt,carrier_type
1500,V444_0,PORT03,PORT09,0.0,5000.00,DTD,27.2024,16.2024,GROUND,1,V88888888_0
1501,V444_0,PORT03,PORT09,0.0,5000.00,DTD,27.2024,9.2024,GROUND,1,V88888888_0
1502,V444_0,PORT03,PORT09,0.0,5000.00,DTD,25.4236,16.4236,GROUND,1,V88888888_0
1503,V444_0,PORT03,PORT09,0.0,5000.00,DTD,25.4236,15.4236,GROUND,1,V88888888_0
1504,V444_0,PORT03,PORT09,0.0,5000.00,DTD,27.2024,13.2024,GROUND,1,V88888888_0
...,...,...,...,...,...,...,...,...,...,...,...
1524,V444_8,PORT11,PORT09,500.0,1999.99,DTD,27.7884,0.2028,AIR,0,V88888888_0
44,V444_8,PORT11,PORT09,2000.0,99999.99,DTD,11.1112,0.0924,AIR,14,V88888888_0
107,V444_8,PORT11,PORT09,2000.0,99999.99,DTD,15.1176,0.1572,AIR,5,V88888888_0
859,V444_8,PORT11,PORT09,2000.0,99999.99,DTD,17.4836,0.1768,AIR,2,V88888888_0


TABLE: Plant Ports
CANDIDATE KEY: ['plant_code', 'port']
Total rows: 22
Unique key combinations: 22
Rows involved in duplicate keys: 0
Is key unique? True
TABLE: Products per Plant
CANDIDATE KEY: ['plant_code', 'product_id']
Total rows: 2036
Unique key combinations: 2036
Rows involved in duplicate keys: 0
Is key unique? True
TABLE: VMI Customers
CANDIDATE KEY: ['plant_code', 'customers']
Total rows: 14
Unique key combinations: 14
Rows involved in duplicate keys: 0
Is key unique? True
TABLE: Warehouse Capacities
CANDIDATE KEY: ['plant_id']
Total rows: 19
Unique key combinations: 19
Rows involved in duplicate keys: 0
Is key unique? True
TABLE: Warehouse Costs
CANDIDATE KEY: ['wh']
Total rows: 19
Unique key combinations: 19
Rows involved in duplicate keys: 0
Is key unique? True


In [31]:
key_summary = pd.DataFrame(key_results)
display(key_summary)

,table,candidate_key,total_rows,unique_keys,duplicate_key_rows,is_unique
0,Orders,order_id,9215,9215,0,True
1,Freight Rates,"carrier, orig_port_cd, dest_port_cd, minm_wgh_...",1540,1258,414,False
2,Plant Ports,"plant_code, port",22,22,0,True
3,Products per Plant,"plant_code, product_id",2036,2036,0,True
4,VMI Customers,"plant_code, customers",14,14,0,True
5,Warehouse Capacities,plant_id,19,19,0,True
6,Warehouse Costs,wh,19,19,0,True


## 2.4 Referential Integrity

In [32]:
def check_referential_integrity(
    child_df,
    child_columns,
    parent_df,
    parent_columns,
    relationship_name
):
    child_keys = (
        child_df[child_columns]
        .drop_duplicates()
        .copy()
    )

    parent_keys = (
        parent_df[parent_columns]
        .drop_duplicates()
        .copy()
    )

    # Standardize temporary column names before joining
    temporary_columns = [
        f"key_{i}" for i in range(len(child_columns))
    ]

    child_keys.columns = temporary_columns
    parent_keys.columns = temporary_columns

    unmatched = (
        child_keys
        .merge(
            parent_keys,
            on=temporary_columns,
            how="left",
            indicator=True
        )
        .query("_merge == 'left_only'")
        .drop(columns="_merge")
    )

    unmatched.columns = child_columns

    orphan_count = len(unmatched)
    passed = orphan_count == 0

    print("=" * 70)
    print(f"RELATIONSHIP: {relationship_name}")
    print("=" * 70)

    print(f"Distinct child keys: {len(child_keys)}")
    print(f"Unmatched keys: {orphan_count}")
    print(f"Integrity check passed? {passed}")

    if orphan_count > 0:
        print("\nUnmatched child keys:")
        display(unmatched)

    return {
        "relationship": relationship_name,
        "child_columns": ", ".join(child_columns),
        "parent_columns": ", ".join(parent_columns),
        "distinct_child_keys": len(child_keys),
        "unmatched_keys": orphan_count,
        "passed": passed
    }

In [33]:
integrity_results = []

integrity_results.append(
    check_referential_integrity(
        child_df=orders,
        child_columns=["plant_code", "origin_port"],
        parent_df=plant_ports,
        parent_columns=["plant_code", "port"],
        relationship_name=(
            "Orders plant and origin port "
            "→ Plant Ports"
        )
    )
)

RELATIONSHIP: Orders plant and origin port → Plant Ports
Distinct child keys: 7
Unmatched keys: 0
Integrity check passed? True


In [34]:
integrity_results.append(
    check_referential_integrity(
        child_df=orders,
        child_columns=["plant_code", "product_id"],
        parent_df=products_per_plant,
        parent_columns=["plant_code", "product_id"],
        relationship_name=(
            "Orders plant and product "
            "→ Products per Plant"
        )
    )
)

RELATIONSHIP: Orders plant and product → Products per Plant
Distinct child keys: 780
Unmatched keys: 0
Integrity check passed? True


In [35]:
integrity_results.append(
    check_referential_integrity(
        child_df=orders,
        child_columns=["plant_code"],
        parent_df=warehouse_capacities,
        parent_columns=["plant_id"],
        relationship_name=(
            "Orders plant "
            "→ Warehouse Capacities"
        )
    )
)

RELATIONSHIP: Orders plant → Warehouse Capacities
Distinct child keys: 7
Unmatched keys: 0
Integrity check passed? True


In [36]:
integrity_results.append(
    check_referential_integrity(
        child_df=orders,
        child_columns=["plant_code"],
        parent_df=warehouse_costs,
        parent_columns=["wh"],
        relationship_name=(
            "Orders plant "
            "→ Warehouse Costs"
        )
    )
)

RELATIONSHIP: Orders plant → Warehouse Costs
Distinct child keys: 7
Unmatched keys: 0
Integrity check passed? True


In [37]:
integrity_results.append(
    check_referential_integrity(
        child_df=orders,
        child_columns=[
            "carrier",
            "origin_port",
            "destination_port",
            "service_level"
        ],
        parent_df=freight_rates,
        parent_columns=[
            "carrier",
            "orig_port_cd",
            "dest_port_cd",
            "svc_cd"
        ],
        relationship_name=(
            "Orders shipping route "
            "→ Freight Rates"
        )
    )
)

RELATIONSHIP: Orders shipping route → Freight Rates
Distinct child keys: 7
Unmatched keys: 3
Integrity check passed? False

Unmatched child keys:


,carrier,origin_port,destination_port,service_level
0,V44_3,PORT09,PORT09,CRF
1,V44_3,PORT04,PORT09,CRF
2,V44_3,PORT05,PORT09,CRF


In [38]:
integrity_summary = pd.DataFrame(integrity_results)
display(integrity_summary)

,relationship,child_columns,parent_columns,distinct_child_keys,unmatched_keys,passed
0,Orders plant and origin port → Plant Ports,"plant_code, origin_port","plant_code, port",7,0,True
1,Orders plant and product → Products per Plant,"plant_code, product_id","plant_code, product_id",780,0,True
2,Orders plant → Warehouse Capacities,plant_code,plant_id,7,0,True
3,Orders plant → Warehouse Costs,plant_code,wh,7,0,True
4,Orders shipping route → Freight Rates,"carrier, origin_port, destination_port, servic...","carrier, orig_port_cd, dest_port_cd, svc_cd",7,3,False


In [39]:
vmi_order_pairs = (
    orders[["plant_code", "customer"]]
    .drop_duplicates()
)

vmi_check = (
    vmi_customers
    .merge(
        vmi_order_pairs,
        left_on=["plant_code", "customers"],
        right_on=["plant_code", "customer"],
        how="left",
        indicator=True
    )
)

vmi_not_in_history = vmi_check[
    vmi_check["_merge"] == "left_only"
]

print(
    "VMI plant-customer pairs not found "
    "in historical orders:",
    len(vmi_not_in_history)
)

display(vmi_not_in_history)

VMI plant-customer pairs not found in historical orders: 14


,plant_code,customers,customer,_merge
0,PLANT02,V5555555555555_16,NaN,left_only
1,PLANT02,V555555555555555_29,NaN,left_only
2,PLANT02,V555555555_3,NaN,left_only
3,PLANT02,V55555555555555_8,NaN,left_only
4,PLANT02,V55555555_9,NaN,left_only
5,PLANT02,V55555_10,NaN,left_only
6,PLANT02,V55555555_5,NaN,left_only
7,PLANT06,V555555555555555_18,NaN,left_only
8,PLANT06,V55555_10,NaN,left_only
9,PLANT10,V555555555555555_29,NaN,left_only


# SUMMARY

In [40]:
tables = {
    "orders": orders,
    "freight_rates": freight_rates,
    "plant_ports": plant_ports,
    "products_per_plant": products_per_plant,
    "vmi_customers": vmi_customers,
    "warehouse_capacities": warehouse_capacities,
    "warehouse_costs": warehouse_costs
}

In [41]:
table_quality_results = []

for table_name, df in tables.items():
    numeric_df = df.select_dtypes(include="number")

    if numeric_df.empty:
        negative_values = 0
    else:
        negative_values = int(
            (numeric_df < 0).sum().sum()
        )

    table_quality_results.append({
        "table": table_name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "missing_values": int(
            df.isna().sum().sum()
        ),
        "duplicate_rows": int(
            df.duplicated().sum()
        ),
        "negative_numeric_values": negative_values
    })

table_quality_summary = pd.DataFrame(
    table_quality_results
)

display(table_quality_summary)

,table,rows,columns,missing_values,duplicate_rows,negative_numeric_values
0,orders,9215,14,0,0,0
1,freight_rates,1540,11,0,3,0
2,plant_ports,22,2,0,0,0
3,products_per_plant,2036,2,0,0,0
4,vmi_customers,14,2,0,0,0
5,warehouse_capacities,19,2,0,0,0
6,warehouse_costs,19,2,0,0,0


In [42]:
print("TABLE-LEVEL QUALITY")
display(table_quality_summary)

print("\nCANDIDATE KEY QUALITY")
display(key_summary)

print("\nREFERENTIAL INTEGRITY")
display(integrity_summary)

TABLE-LEVEL QUALITY


,table,rows,columns,missing_values,duplicate_rows,negative_numeric_values
0,orders,9215,14,0,0,0
1,freight_rates,1540,11,0,3,0
2,plant_ports,22,2,0,0,0
3,products_per_plant,2036,2,0,0,0
4,vmi_customers,14,2,0,0,0
5,warehouse_capacities,19,2,0,0,0
6,warehouse_costs,19,2,0,0,0



CANDIDATE KEY QUALITY


,table,candidate_key,total_rows,unique_keys,duplicate_key_rows,is_unique
0,Orders,order_id,9215,9215,0,True
1,Freight Rates,"carrier, orig_port_cd, dest_port_cd, minm_wgh_...",1540,1258,414,False
2,Plant Ports,"plant_code, port",22,22,0,True
3,Products per Plant,"plant_code, product_id",2036,2036,0,True
4,VMI Customers,"plant_code, customers",14,14,0,True
5,Warehouse Capacities,plant_id,19,19,0,True
6,Warehouse Costs,wh,19,19,0,True



REFERENTIAL INTEGRITY


,relationship,child_columns,parent_columns,distinct_child_keys,unmatched_keys,passed
0,Orders plant and origin port → Plant Ports,"plant_code, origin_port","plant_code, port",7,0,True
1,Orders plant and product → Products per Plant,"plant_code, product_id","plant_code, product_id",780,0,True
2,Orders plant → Warehouse Capacities,plant_code,plant_id,7,0,True
3,Orders plant → Warehouse Costs,plant_code,wh,7,0,True
4,Orders shipping route → Freight Rates,"carrier, origin_port, destination_port, servic...","carrier, orig_port_cd, dest_port_cd, svc_cd",7,3,False


In [43]:
total_missing = (
    table_quality_summary["missing_values"].sum()
)

total_duplicates = (
    table_quality_summary["duplicate_rows"].sum()
)

failed_keys = (
    (~key_summary["is_unique"]).sum()
)

failed_relationships = (
    (~integrity_summary["passed"]).sum()
)

print("DATA QUALITY SUMMARY")
print("=" * 70)
print(f"Total missing values: {total_missing}")
print(f"Total duplicate rows: {total_duplicates}")
print(f"Non-unique candidate keys: {failed_keys}")
print(
    "Relationships with unmatched keys:",
    failed_relationships
)

DATA QUALITY SUMMARY
Total missing values: 0
Total duplicate rows: 3
Non-unique candidate keys: 1
Relationships with unmatched keys: 1
